<a href="https://colab.research.google.com/github/riyajain26/eeg-digit-classification/blob/main/notebooks/Colab_Train_Evaluate_Test_Stage1_2B20pct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab — Train, Evaluate, and Test All Models (Stage 1)

## Before running

- **Enable GPU**: `Runtime -> Change runtime type -> GPU`
- **Upload data first**: this notebook expects `data/` and
  `models/preprocessing/` already present in your Drive project folder
  (produced by the local data-pipeline notebook, uploaded manually).

## Data flow

Copies `data/` and `models/preprocessing/` from Drive to Colab's LOCAL
disk once (Drive/FUSE caused an out-of-memory crash during heavy write
access earlier — reads are less risky, but we copy once up front rather
than repeatedly reading through FUSE during training regardless).
Everything else (checkpoints, results) is written locally, then synced
back to Drive at the end.

## 1. Mount Drive & Copy Data to Local Disk

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/eeg-digit-classification')   # adjust if yours differs
LOCAL_DIR = Path('/content/eeg_work')

shutil.copytree(DRIVE_DIR / "data", LOCAL_DIR / "data", dirs_exist_ok=True)
shutil.copytree(DRIVE_DIR / "models" / "preprocessing", LOCAL_DIR / "models" / "preprocessing", dirs_exist_ok=True)
print("Copied data/ and models/preprocessing/ to local disk.")

Mounted at /content/drive
Copied data/ and models/preprocessing/ to local disk.


## 2. Clone Repo & Install Dependencies

Replace `REPO_URL` with your actual GitHub URL.

In [2]:
REPO_URL = "https://github.com/riyajain26/eeg-digit-classification.git"

%cd /content
!rm -rf eeg-digit-classification # Remove existing directory to allow fresh clone
!git clone {REPO_URL}
%cd eeg-digit-classification

!pip install -q -r src/requirements.txt

import torch
print(f"\nGPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected - Runtime > Change runtime type > GPU, then re-run.")

/content
Cloning into 'eeg-digit-classification'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 140 (delta 55), reused 136 (delta 55), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 520.08 KiB | 334.00 KiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/eeg-digit-classification

GPU available: False


## 3. Config Helper — Points at Local Disk, Not Drive

In [3]:
import sys
sys.path.insert(0, '/content/eeg-digit-classification')

from src.config import build_config

def make_local_config(**kwargs):
    cfg = build_config(**kwargs)
    cfg.data.data_root = LOCAL_DIR / "data"
    cfg.model.model_root = LOCAL_DIR / "models"
    return cfg

def sync_to_drive():
    shutil.copytree(LOCAL_DIR / "data", DRIVE_DIR / "data", dirs_exist_ok=True)
    shutil.copytree(LOCAL_DIR / "models", DRIVE_DIR / "models", dirs_exist_ok=True)
    print("Synced to Drive.")

DATASET_VARIANT = "2B"
SUBSAMPLE_FRACTION = 0.20
STAGE = "stage1"

# Verify the expected input files are actually present before training anything
test_cfg = make_local_config(dataset_variant=DATASET_VARIANT, subsample_fraction=SUBSAMPLE_FRACTION,
                              stage=STAGE, model_name="lda")
required = [
    test_cfg.data.features_dir / "train_features.h5",
    test_cfg.data.features_dir / "test_features.h5",
    test_cfg.data.filtered_dir / "train_filtered.h5",
    test_cfg.data.filtered_dir / "test_filtered.h5",
    test_cfg.model.preprocessing_dir(test_cfg.data.variant_tag) / "normalization_params.npz",
]
for path in required:
    print(f"{'OK' if path.exists() else 'MISSING'}: {path}")

OK: /content/eeg_work/data/processed/2B_frac0.20/features/train_features.h5
OK: /content/eeg_work/data/processed/2B_frac0.20/features/test_features.h5
OK: /content/eeg_work/data/processed/2B_frac0.20/filtered/train_filtered.h5
OK: /content/eeg_work/data/processed/2B_frac0.20/filtered/test_filtered.h5
OK: /content/eeg_work/models/preprocessing/2B_frac0.20/normalization_params.npz


## 4. Train + Evaluate on Val — All 4 Models

Only the classical models' permutation tests should be fast now (SVM uses
a capped LinearSVC surrogate for permutation testing specifically - see
`models/factory.py`). EEGNet training is the slow, GPU-bound part.

In [5]:
from src.pipeline import (run_phase5_6_model,)

MODELS_TO_RUN = ["lda", "svm", "random_forest", "eegnet"]
#MODELS_TO_RUN = ["random_forest", "eegnet"]
#MODELS_TO_RUN = ["eegnet"]
val_results = {}


for model_name in MODELS_TO_RUN:
    print(f"\n{'='*60}\n{model_name}\n{'='*60}")
    cfg = make_local_config(dataset_variant=DATASET_VARIANT, subsample_fraction=SUBSAMPLE_FRACTION,
                             stage=STAGE, model_name=model_name)
    val_results[model_name] = run_phase5_6_model(cfg, run_permutation=False)

sync_to_drive()
print("\nAll 4 models trained/evaluated on val, synced to Drive.")


lda
Phase 5 [stage1/lda]: training...
{'model': 'lda', 'accuracy': 0.518859649122807, 'precision': 0.5207780725022104, 'recall': 0.5148601398601399, 'f1': 0.5178021978021978}
Model saved to /content/eeg_work/models/checkpoints/2B_frac0.20__stage1__lda/lda.pkl
Skipping permutation test (run_permutation=False).
Results saved to /content/eeg_work/models/results/2B_frac0.20__stage1__lda/results.json

svm
Phase 5 [stage1/svm]: training...
{'model': 'svm', 'accuracy': 0.5289473684210526, 'precision': 0.5311942959001783, 'recall': 0.5209790209790209, 'f1': 0.5260370697263901}
Model saved to /content/eeg_work/models/checkpoints/2B_frac0.20__stage1__svm/svm.pkl
Skipping permutation test (run_permutation=False).
Results saved to /content/eeg_work/models/results/2B_frac0.20__stage1__svm/results.json

random_forest
Phase 5 [stage1/random_forest]: training...
{'model': 'random_forest', 'accuracy': 0.5513157894736842, 'precision': 0.557345971563981, 'recall': 0.513986013986014, 'f1': 0.534788540245

## 5. Phase 7 — Evaluate on Held-Out Test Set

Loads each already-trained checkpoint (does NOT retrain) and evaluates on
test. This is the final, only-look-at-once number for each model.

In [6]:
from src.pipeline import run_phase7_evaluate_on_test

test_results = {}

for model_name in MODELS_TO_RUN:
    print(f"\n{'='*60}\n{model_name} - TEST\n{'='*60}")
    cfg = make_local_config(dataset_variant=DATASET_VARIANT, subsample_fraction=SUBSAMPLE_FRACTION,
                             stage=STAGE, model_name=model_name)
    test_results[model_name] = run_phase7_evaluate_on_test(cfg)

sync_to_drive()
print("\nAll 4 models evaluated on test, synced to Drive.")


lda - TEST
Phase 7 [stage1/lda] TEST metrics: {'model': 'lda', 'accuracy': 0.5069375619425174, 'precision': 0.5538461538461539, 'recall': 0.03592814371257485, 'f1': 0.06747891283973759}
Val vs. test: {'val_accuracy': 0.518859649122807, 'test_accuracy': 0.5069375619425174, 'gap': 0.011922087180289642}
Test results saved to /content/eeg_work/models/results/2B_frac0.20__stage1__lda/test_results.json

svm - TEST
Phase 7 [stage1/svm] TEST metrics: {'model': 'svm', 'accuracy': 0.5044598612487612, 'precision': 0.5018518518518519, 'recall': 0.27045908183632733, 'f1': 0.35149156939040205}
Val vs. test: {'val_accuracy': 0.5289473684210526, 'test_accuracy': 0.5044598612487612, 'gap': 0.024487507172291423}
Test results saved to /content/eeg_work/models/results/2B_frac0.20__stage1__svm/test_results.json

random_forest - TEST
Phase 7 [stage1/random_forest] TEST metrics: {'model': 'random_forest', 'accuracy': 0.5267591674925669, 'precision': 0.5217391304347826, 'recall': 0.562874251497006, 'f1': 0.5

## 6. Summary Table — Val vs. Test, All Models

In [10]:
import pandas as pd

rows = []
for model_name in MODELS_TO_RUN:
    val_acc = val_results[model_name]["metrics"]["accuracy"]
    test_acc = test_results[model_name]["test_metrics"]["accuracy"]
    perm = val_results[model_name]["permutation_test"]
    rows.append({
        "model": model_name,
        "val_accuracy": val_acc,
        "test_accuracy": test_acc,
        "val_test_gap": val_acc - test_acc,
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

print("\nWatch for: large val_test_gap (overfitting to val), or permutation_gap_over_std")
print("near/below 1 (result indistinguishable from chance for that model).")

        model  val_accuracy  test_accuracy  val_test_gap
          lda      0.518860       0.506938      0.011922
          svm      0.528947       0.504460      0.024488
random_forest      0.551316       0.526759      0.024557
       eegnet      0.768860       0.598117      0.170743

Watch for: large val_test_gap (overfitting to val), or permutation_gap_over_std
near/below 1 (result indistinguishable from chance for that model).


# Stage 1 Findings Summary (20% MindBigData2023 MNIST-2B)

**Task**: binary EEG classification — blank vs. digit-stimulus trials.

**Pipeline**: leakage-safe (session/block-aware) split → filtering (1-40Hz)
→ robust normalization → artifact flagging → classical baselines (Path A)
+ EEGNet (Path B), fully parameterized via src/.

## Results (val / test accuracy)

| model         | val    | test   | gap    |
|---------------|--------|--------|--------|
| LDA           | 0.519  | 0.507  | 0.012  |
| SVM           | 0.529  | 0.504  | 0.024  |
| RandomForest  | 0.551  | 0.527  | 0.025  |
| EEGNet        | 0.769  | 0.598  | 0.171  |

## Key findings

- **EEGNet substantially outperforms every classical baseline** on both val
  and test, confirming raw-signal deep learning captures structure
  hand-crafted features (band power, statistics) can't.
- **Permutation testing (development runs, local data) confirmed genuine
  signal, not chance**: RandomForest ~2.5x its shuffled-label std, EEGNet
  ~30x — both well above the noise floor.
- **EEGNet's val-test gap (17pp) is far larger than classical models'
  (1-2.5pp)** — the standout open question. Leading hypothesis: raw-signal
  models are more sensitive to train/test distribution shift (test is a
  structurally separate source file, likely different sessions) than
  hand-crafted features, which average out session-specific noise.
- **Artifact rate varied notably by run** (~13% locally vs. ~55% on a
  fresh Colab stream) — likely reflects genuine session-to-session
  heterogeneity given percentile-based thresholds should otherwise
  self-calibrate. Not yet root-caused.

## Open items for next round
- Investigate EEGNet's train-test generalization gap (session-level
  breakdown, regularization, more data via full-scale run)
- Root-cause the artifact-rate variance across runs
- Feature importance / channel analysis (deferred from Phase 5)